In [1]:
# Chain Of Thought Prompting
from dotenv import load_dotenv
from openai import OpenAI
import requests
from pydantic import BaseModel, Field
from typing import Optional
import json
import os
from google import genai
import re

In [2]:
load_dotenv()

True

In [3]:
client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [4]:
def run_command(cmd: str):
    result = os.system(cmd)
    return result

In [5]:
def get_weather(city: str):
    url = f"https://wttr.in/{city.lower()}?format=%C+%t"
    response = requests.get(url)

    if response.status_code == 200:
        return f"The weather in {city} is {response.text}"
    
    return "Something went wrong"

In [6]:
available_tools = {
    "get_weather": get_weather,
    "run_command": run_command
}

In [7]:
SYSTEM_PROMPT = """
    You're an expert AI Assistant in resolving user queries using chain of thought.
    You work on START, PLAN and OUPUT steps.
    You need to first PLAN what needs to be done. The PLAN can be multiple steps.
    Once you think enough PLAN has been done, finally you can give an OUTPUT.
    You can also call a tool if required from the list of available tools.
    for every tool call wait for the observe step which is the output from the called tool.

    Rules:
    - Strictly Follow the given JSON output format
    - Only run one step at a time.
    - The sequence of steps is START (where user gives an input), PLAN (That can be multiple times) and finally OUTPUT (which is going to the displayed to the user).

    Output JSON Format:
    { "step": "START" | "PLAN" | "OUTPUT" | "TOOL", "content": "string", "tool": "string", "input": "string" }

    Available Tools:
    - get_weather(city: str): Takes city name as an input string and returns the weather info about the city.
    - run_command(cmd: str): Takes a system linux command as string and executes the command on user's system and returns the output from that command
    
    Example 1:
    START: Hey, Can you solve 2 + 3 * 5 / 10
    PLAN: { "step": "PLAN": "content": "Seems like user is interested in math problem" }
    PLAN: { "step": "PLAN": "content": "looking at the problem, we should solve this using BODMAS method" }
    PLAN: { "step": "PLAN": "content": "Yes, The BODMAS is correct thing to be done here" }
    PLAN: { "step": "PLAN": "content": "first we must multiply 3 * 5 which is 15" }
    PLAN: { "step": "PLAN": "content": "Now the new equation is 2 + 15 / 10" }
    PLAN: { "step": "PLAN": "content": "We must perform divide that is 15 / 10  = 1.5" }
    PLAN: { "step": "PLAN": "content": "Now the new equation is 2 + 1.5" }
    PLAN: { "step": "PLAN": "content": "Now finally lets perform the add 3.5" }
    PLAN: { "step": "PLAN": "content": "Great, we have solved and finally left with 3.5 as ans" }
    OUTPUT: { "step": "OUTPUT": "content": "3.5" }

    Example 2:
    START: What is the weather of Delhi?
    PLAN: { "step": "PLAN": "content": "Seems like user is interested in getting weather of Delhi in India" }
    PLAN: { "step": "PLAN": "content": "Lets see if we have any available tool from the list of available tools" }
    PLAN: { "step": "PLAN": "content": "Great, we have get_weather tool available for this query." }
    PLAN: { "step": "PLAN": "content": "I need to call get_weather tool for delhi as input for city" }
    PLAN: { "step": "TOOL": "tool": "get_weather", "input": "delhi" }
    PLAN: { "step": "OBSERVE": "tool": "get_weather", "output": "The temp of delhi is cloudy with 20 C" }
    PLAN: { "step": "PLAN": "content": "Great, I got the weather info about delhi" }
    OUTPUT: { "step": "OUTPUT": "content": "The cuurent weather in delhi is 20 C with some cloudy sky." }
    
"""

print("\n\n\n")

In [10]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/gemini-embedding-2-

In [ ]:
class MyOutputFormat(BaseModel):
    step: str = Field(..., description="The ID of the step. Example: PLAN, OUTPUT, TOOL, etc")
    content: Optional[str] = Field(None, description="The optional string content for the step")
    tool: Optional[str] = Field(None, description="The ID of the tool to call.")
    input: Optional[str] = Field(None, description="The input params for the tool")

message_history = [
    { "role": "user", "parts": [{"text": f"SYSTEM: {SYSTEM_PROMPT}"}] },
    { "role": "model", "parts": [{"text": "Understood."}] },
]

while True:
    user_query = input("Send Input: ")
    message_history.append({ "role": "user", "parts": [{"text": user_query}] })

    
    while True:  # ← indented inside outer loop
        response = client.models.generate_content(
            model="gemini-3.1-flash-lite-preview",
            contents=message_history,
            config={
                "response_mime_type": "application/json",
                "response_schema": MyOutputFormat,
                "max_output_tokens": 1024,
            }
        )

        raw_result = response.text
        if not raw_result:
            continue
            
        message_history.append({"role": "model", "parts": [{"text": raw_result}]})
        match = re.search(r'\{.*?\}', raw_result, re.DOTALL)
        if match:
            parsed_result = MyOutputFormat(**json.loads(raw_result))

        if parsed_result.step == "START":
            print("->", parsed_result.content)
            continue

        if parsed_result.step == "TOOL":
            tool_to_call = parsed_result.tool
            tool_input = parsed_result.input
            print(f"-> TOOL: {tool_to_call} ({tool_input})")
            tool_response = available_tools[tool_to_call](tool_input)
            print(f"-> RESULT: {tool_to_call} ({tool_input}) = {tool_response}")
            message_history.append({ 
                "role": "user", 
                "parts": [{"text": json.dumps(
                    {"step": "OBSERVE", "tool": tool_to_call, "input": tool_input, "output": tool_response}
                )}]
            })
            continue

        if parsed_result.step == "PLAN":
            print("->", parsed_result.content)
            continue

        if parsed_result.step == "OUTPUT":
            print("->", parsed_result.content)
            break

print("\n\n\n")

-> I will create a directory named todo_app and generate three files: index.html, style.css, and script.js, which collectively implement a functional CRUD todo application.
-> TOOL: run_command (mkdir todo_app && cd todo_app && touch index.html style.css script.js && echo '<html><body><div id="app"><h1>Todo App</h1><input id="todoInput"><button onclick="addTodo()">Add</button><ul id="todoList"></ul></div><script src="script.js"></script></body></html>' > index.html && echo 'body { font-family: sans-serif; }' > style.css && echo 'let todos = []; function render() { const list = document.getElementById("todoList"); list.innerHTML = ""; todos.forEach((t, i) => { list.innerHTML += `<li>${t} <button onclick="deleteTodo(${i})">Delete</button></li>`; }); } function addTodo() { const input = document.getElementById("todoInput"); todos.push(input.value); input.value = ""; render(); } function deleteTodo(i) { todos.splice(i, 1); render(); }' > script.js)
-> RESULT: run_command (mkdir todo_app &&